# Incident Response Runbook: Collection - Data from Local System

**Tactic:** Collection
**Technique:** Data from Local System (T1005)
**Severity:** HIGH

## Overview

This runbook provides step-by-step incident response procedures for Data from Local System activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime
import sys
import os

# Add the project root to the Python path
sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), '..', '..')))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_response import CrowdStrikeResponse
from iris.iris_integration import IRISIntegration
from misp.misp_integration import MISPIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Initialize integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeResponse()
iris = IRISIntegration()
misp = MISPIntegration()
shuffle = ShuffleIntegration()

# Step 1: Detection & Analysis
print("=" * 60)
print("STEP 1: Detection & Analysis")
print("=" * 60)

detection_time = datetime.now().isoformat()

# Query Splunk for data collection indicators
print(f"\n[QUERY] Searching Splunk for data collection indicators...")
splunk_query = '''
index=* (sourcetype=WinEventLog:Security EventCode=4663 OR EventCode=4656 OR EventCode=4659)
OR (sourcetype=linux_secure "access" OR "read" OR "cat" OR "grep" sensitive_files)
OR (sourcetype=fs_notification "read" OR "access" (file_path="*.docx" OR file_path="*.xlsx" OR file_path="*.pdf" OR file_path="*.txt"))
OR (sourcetype=WinEventLog:Security EventCode=5145 (ShareName!="*IPC$") ObjectName!="*\\*")
| eval file_path=coalesce(ObjectName, file_path, FileName)
| eval sensitive_files=if(match(file_path, ".*\\\\(Users|Documents|Desktop)\\\\.*") OR match(file_path, ".*\\.(docx|xlsx|pdf|txt|key|pem|cer)$"), "true", "false")
| where sensitive_files="true"
| stats count by host, user, file_path, _time
| where count > 3
'''

try:
    splunk_results = splunk.search_events(splunk_query, timeframe="-24h")
    print(f"   Found {len(splunk_results)} potential data collection events")
except Exception as e:
    print(f"   Splunk query failed: {e}")
    splunk_results = []

# Extract affected systems and data collection activities
affected_systems = []
data_collection_activities = []
splunk_indicators = []

for event in splunk_results:
    system_info = {
        'hostname': event.get('host', 'unknown'),
        'user': event.get('user', 'unknown'),
        'file_path': event.get('file_path', ''),
        'access_count': event.get('count', 0),
        'last_seen': event.get('_time', detection_time)
    }
    affected_systems.append(system_info)

    # Extract indicators
    if event.get('file_path'):
        splunk_indicators.append({
            'type': 'file',
            'value': event.get('file_path'),
            'context': f"Sensitive file accessed by {event.get('user', 'unknown')} on {event.get('host', 'unknown')}"
        })

# Query CrowdStrike for data collection detections
print(f"\n[QUERY] Checking CrowdStrike for data collection detections...")
try:
    cs_detections = crowdstrike.get_detections(
        filter="technique:'Data from Local System'",
        start_time="-24h"
    )
    cs_affected_hosts = []
    for detection in cs_detections:
        host_info = {
            'device_id': detection.get('device_id', ''),
            'hostname': detection.get('hostname', ''),
            'detection_id': detection.get('detection_id', ''),
            'technique': detection.get('technique', ''),
            'severity': detection.get('severity', 0),
            'collected_files': detection.get('collected_files', [])
        }
        cs_affected_hosts.append(host_info)

        # Merge with Splunk data
        existing_system = next((s for s in affected_systems if s['hostname'] == host_info['hostname']), None)
        if existing_system:
            existing_system.update(host_info)
        else:
            affected_systems.append(host_info)

    print(f"   Found {len(cs_affected_hosts)} CrowdStrike data collection detections")
except Exception as e:
    print(f"   CrowdStrike query failed: {e}")
    cs_affected_hosts = []

# Enrich with threat intelligence from MISP
print(f"\n[ENRICHMENT] Checking MISP for related threat intelligence...")
misp_results = []
try:
    for indicator in splunk_indicators:
        misp_search = misp.search_iocs(indicator['value'])
        if misp_search:
            misp_results.extend(misp_search)
            print(f"   Found threat intelligence for {indicator['value']}")
except Exception as e:
    print(f"   MISP enrichment failed: {e}")

# Create IRIS case
print(f"\n[CASE] Creating IRIS incident case...")
try:
    incident_data = {
        'title': f'Data Collection Incident - {len(affected_systems)} systems',
        'description': f'Detected unauthorized data collection activities on {len(affected_systems)} systems',
        'severity': 'HIGH',
        'tactic': 'Collection',
        'technique': 'Data from Local System (T1005)',
        'affected_systems': affected_systems,
        'indicators': splunk_indicators,
        'threat_intel': misp_results,
        'detection_time': detection_time
    }
    incident_id = iris.create_case(incident_data)
    print(f"   Created IRIS case: {incident_id}")
except Exception as e:
    print(f"   IRIS case creation failed: {e}")
    incident_id = f"TEMP-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

print(f"\n Detection complete:")
print(f"  - {len(affected_systems)} affected systems identified")
print(f"  - {len(splunk_indicators)} data collection indicators found")
print(f"  - {len(misp_results)} threat intelligence matches")
print(f"  - IRIS case {incident_id} created")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment
print("\n" + "=" * 60)
print("STEP 2: Containment")
print("=" * 60)

containment_time = datetime.now().isoformat()
isolated_hosts = []
disabled_accounts = []
blocked_files = []

# Isolate affected systems
print(f"\n[ACTION] Isolating affected systems...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            isolation_result = crowdstrike.isolate_host(system['device_id'])
            if isolation_result.get('status') == 'isolated':
                isolated_hosts.append(system['hostname'])
                print(f"   Isolated host: {system['hostname']}")
    except Exception as e:
        print(f"   Host isolation failed for {system.get('hostname', 'unknown')}: {e}")

# Disable suspicious user accounts
print(f"\n[ACTION] Disabling suspicious user accounts...")
unique_users = set()
for system in affected_systems:
    if system.get('user') and system['user'] != 'unknown':
        unique_users.add(system['user'])

for user in unique_users:
    try:
        disable_result = shuffle.disable_user_account(user)
        if disable_result.get('status') == 'disabled':
            disabled_accounts.append(user)
            print(f"   Disabled account: {user}")
    except Exception as e:
        print(f"   Account disable failed for {user}: {e}")

# Block access to sensitive files
print(f"\n[ACTION] Blocking access to sensitive files...")
unique_files = set()
for system in affected_systems:
    if system.get('file_path'):
        unique_files.add(system['file_path'])

for file_path in unique_files:
    try:
        block_result = shuffle.block_file_access(file_path)
        if block_result.get('status') == 'blocked':
            blocked_files.append(file_path)
            print(f"   Blocked access to: {file_path}")
    except Exception as e:
        print(f"   File access blocking failed for {file_path}: {e}")

# Enable enhanced monitoring
print(f"\n[ACTION] Enabling enhanced monitoring...")
monitoring_rules = []
try:
    # Enable CrowdStrike data collection monitoring
    cs_monitoring = crowdstrike.enable_enhanced_monitoring('data_collection')
    if cs_monitoring.get('status') == 'enabled':
        monitoring_rules.append('CrowdStrike data collection monitoring')
        print(f"   Enabled CrowdStrike data collection monitoring")

    # Enable Splunk file access correlation rules
    splunk_monitoring = splunk.enable_correlation_rule('sensitive_file_access')
    if splunk_monitoring.get('status') == 'enabled':
        monitoring_rules.append('Splunk file access correlation')
        print(f"   Enabled Splunk file access correlation rules")
except Exception as e:
    print(f"   Enhanced monitoring setup failed: {e}")

# Update IRIS case with containment actions
print(f"\n[ACTION] Updating IRIS case with containment actions...")
try:
    containment_summary = {
        'isolated_hosts': len(isolated_hosts),
        'disabled_accounts': len(disabled_accounts),
        'blocked_files': len(blocked_files),
        'monitoring_enabled': len(monitoring_rules),
        'containment_time': containment_time
    }
    iris.update_case(incident_id, 'containment_complete', containment_summary)
    print(f"   Updated IRIS case {incident_id} with containment results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

print(f"\n Containment complete:")
print(f"  - {len(isolated_hosts)} hosts isolated")
print(f"  - {len(disabled_accounts)} accounts disabled")
print(f"  - {len(blocked_files)} sensitive files blocked")
print(f"  - {len(monitoring_rules)} enhanced monitoring rules enabled")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication
print("\n" + "=" * 60)
print("STEP 3: Eradication")
print("=" * 60)

removed_collectors = []
encrypted_data = []
cleaned_logs = []

# Remove data collection tools
print(f"\n[ACTION] Removing data collection tools...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            collector_removal = crowdstrike.remove_data_collectors(system['device_id'])
            if collector_removal.get('status') == 'removed':
                removed_collectors.extend(collector_removal.get('removed_tools', []))
                print(f"   Removed {len(collector_removal.get('removed_tools', []))} data collection tools from {system['hostname']}")
    except Exception as e:
        print(f"   Data collector removal failed for {system.get('hostname', 'unknown')}: {e}")

# Encrypt sensitive data at rest
print(f"\n[ACTION] Encrypting sensitive data...")
for file_path in blocked_files:
    try:
        encrypt_result = shuffle.encrypt_sensitive_file(file_path)
        if encrypt_result.get('status') == 'encrypted':
            encrypted_data.append(file_path)
            print(f"   Encrypted sensitive file: {file_path}")
    except Exception as e:
        print(f"   File encryption failed for {file_path}: {e}")

# Clean access logs
print(f"\n[ACTION] Cleaning access logs...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            log_cleaning = crowdstrike.clean_file_access_logs(system['device_id'])
            if log_cleaning.get('status') == 'cleaned':
                cleaned_logs.append(system['hostname'])
                print(f"   Cleaned file access logs on {system['hostname']}")
    except Exception as e:
        print(f"   Log cleaning failed for {system.get('hostname', 'unknown')}: {e}")

# Remove unauthorized copies
print(f"\n[ACTION] Removing unauthorized data copies...")
removed_copies = []
for system in affected_systems:
    try:
        if 'device_id' in system:
            copy_removal = crowdstrike.remove_unauthorized_copies(system['device_id'])
            if copy_removal.get('status') == 'removed':
                removed_copies.extend(copy_removal.get('removed_files', []))
                print(f"   Removed {len(copy_removal.get('removed_files', []))} unauthorized copies from {system['hostname']}")
    except Exception as e:
        print(f"   Unauthorized copy removal failed for {system.get('hostname', 'unknown')}: {e}")

# Verify threat removal
print(f"\n[ACTION] Verifying threat removal...")
verification_results = []
for system in affected_systems:
    try:
        if 'device_id' in system:
            verify_result = crowdstrike.verify_data_collection_removal(system['device_id'])
            verification_results.append({
                'system': system['hostname'],
                'status': 'clean' if verify_result.get('collection_detected', True) == False else 'threats_remaining',
                'remaining_indicators': verify_result.get('remaining_indicators', 0)
            })
            status = "" if verify_result.get('collection_detected', True) == False else ""
            print(f"  {status} Verification for {system['hostname']}: {verify_result.get('remaining_indicators', 0)} collection indicators remaining")
    except Exception as e:
        print(f"   Verification failed for {system.get('hostname', 'unknown')}: {e}")

print(f"\n Eradication complete:")
print(f"  - {len(removed_collectors)} data collection tools removed")
print(f"  - {len(encrypted_data)} sensitive files encrypted")
print(f"  - {len(cleaned_logs)} systems had logs cleaned")
print(f"  - {len(removed_copies)} unauthorized copies removed")
print(f"  - {len([v for v in verification_results if v['status'] == 'clean'])} systems verified clean")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery
print("\n" + "=" * 60)
print("STEP 4: Recovery")
print("=" * 60)

restored_systems = []
restored_access = []
validated_functions = []

# Restore systems from clean backups
print(f"\n[ACTION] Restoring systems from clean backups...")
for system in affected_systems:
    try:
        if 'device_id' in system:
            restore_result = crowdstrike.restore_from_backup(system['device_id'])
            if restore_result.get('status') == 'restored':
                restored_systems.append(system['hostname'])
                print(f"   Restored {system['hostname']} from clean backup")
    except Exception as e:
        print(f"   System restoration failed for {system.get('hostname', 'unknown')}: {e}")

# Restore legitimate file access
print(f"\n[ACTION] Restoring legitimate file access...")
for file_path in blocked_files:
    try:
        # Check if file access should be restored
        access_check = shuffle.validate_file_access(file_path)
        if access_check.get('legitimate', False):
            restore_result = shuffle.restore_file_access(file_path)
            if restore_result.get('status') == 'restored':
                restored_access.append(file_path)
                print(f"   Restored legitimate access to: {file_path}")
        else:
            print(f"   File {file_path} flagged as sensitive, access not restored")
    except Exception as e:
        print(f"   File access restoration failed for {file_path}: {e}")

# Re-enable legitimate user accounts
print(f"\n[ACTION] Re-enabling legitimate user accounts...")
reenabled_accounts = []
for account in disabled_accounts:
    try:
        # Validate account legitimacy
        account_check = shuffle.validate_user_account(account)
        if account_check.get('legitimate', False):
            enable_result = shuffle.enable_user_account(account)
            if enable_result.get('status') == 'enabled':
                reenabled_accounts.append(account)
                print(f"   Re-enabled account: {account}")
        else:
            print(f"   Account {account} flagged as compromised, not re-enabled")
    except Exception as e:
        print(f"   Account re-enabling failed for {account}: {e}")

# Validate system functionality
print(f"\n[ACTION] Validating system functionality...")
functional_tests = []
for system in affected_systems:
    try:
        if 'device_id' in system:
            test_result = crowdstrike.perform_functional_testing(system['device_id'])
            functional_tests.append({
                'system': system['hostname'],
                'tests_passed': test_result.get('tests_passed', 0),
                'total_tests': test_result.get('total_tests', 0)
            })
            pass_rate = test_result.get('tests_passed', 0) / max(test_result.get('total_tests', 1), 1) * 100
            status = "" if pass_rate >= 95 else ""
            print(f"  {status} Functional tests for {system['hostname']}: {test_result.get('tests_passed', 0)}/{test_result.get('total_tests', 0)} passed ({pass_rate:.1f}%)")
    except Exception as e:
        print(f"   Functional testing failed for {system.get('hostname', 'unknown')}: {e}")

# Monitor for residual issues
print(f"\n[ACTION] Establishing monitoring for residual issues...")
monitoring_alerts = []
try:
    # Set up continuous monitoring
    monitoring_setup = splunk.setup_continuous_monitoring('data_collection')
    if monitoring_setup.get('status') == 'enabled':
        monitoring_alerts.append('Splunk continuous monitoring')
        print(f"   Established continuous Splunk monitoring")

    cs_monitoring = crowdstrike.setup_residual_monitoring('data_collection')
    if cs_monitoring.get('status') == 'enabled':
        monitoring_alerts.append('CrowdStrike residual monitoring')
        print(f"   Established CrowdStrike residual monitoring")
except Exception as e:
    print(f"   Residual monitoring setup failed: {e}")

print(f"\n Recovery complete:")
print(f"  - {len(restored_systems)} systems restored from backup")
print(f"  - {len(restored_access)} legitimate file accesses restored")
print(f"  - {len(reenabled_accounts)} legitimate accounts re-enabled")
print(f"  - {len([f for f in functional_tests if f['tests_passed'] == f['total_tests']])} systems passed all functional tests")
print(f"  - {len(monitoring_alerts)} residual monitoring systems established")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident
print("\n" + "=" * 60)
print("STEP 5: Post-Incident")
print("=" * 60)

# Update IRIS case with eradication results
print(f"\n[ACTION] Updating IRIS case with eradication results...")
try:
    eradication_summary = {
        'removed_collectors': len(removed_collectors),
        'encrypted_data': len(encrypted_data),
        'cleaned_logs': len(cleaned_logs),
        'removed_copies': len(removed_copies),
        'verification_results': verification_results
    }
    iris.update_case(incident_id, 'eradication_complete', eradication_summary)
    print(f"   Updated IRIS case {incident_id} with eradication results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

# Update IRIS case with recovery results
print(f"\n[ACTION] Updating IRIS case with recovery results...")
try:
    recovery_summary = {
        'restored_systems': len(restored_systems),
        'restored_access': len(restored_access),
        'reenabled_accounts': len(reenabled_accounts),
        'functional_tests': functional_tests,
        'monitoring_alerts': len(monitoring_alerts)
    }
    iris.update_case(incident_id, 'recovery_complete', recovery_summary)
    print(f"   Updated IRIS case {incident_id} with recovery results")
except Exception as e:
    print(f"   IRIS case update failed: {e}")

# Generate incident report
print(f"\n[ACTION] Generating incident report...")
try:
    incident_report = {
        'incident_id': incident_id,
        'technique': 'Collection: Data from Local System',
        'detection_time': detection_time,
        'affected_systems': len(affected_systems),
        'timeline': {
            'detection': detection_time,
            'containment': containment_time,
            'eradication': datetime.now().isoformat(),
            'recovery': datetime.now().isoformat()
        },
        'actions_taken': {
            'detection': f"Found {len(affected_systems)} systems with data collection activities",
            'containment': f"Isolated {len(isolated_hosts)} hosts, disabled {len(disabled_accounts)} accounts",
            'eradication': f"Removed {len(removed_collectors)} tools, encrypted {len(encrypted_data)} files",
            'recovery': f"Restored {len(restored_systems)} systems, re-enabled {len(reenabled_accounts)} accounts"
        },
        'indicators': splunk_indicators,
        'threat_intel': misp_results,
        'lessons_learned': [
            "Implement stricter file access monitoring and DLP controls",
            "Regular auditing of sensitive file access patterns",
            "Enhanced detection rules for data collection activities"
        ]
    }
    iris.generate_report(incident_id, incident_report)
    print(f"   Generated comprehensive incident report for case {incident_id}")
except Exception as e:
    print(f"   Report generation failed: {e}")

# Share IOCs with MISP
print(f"\n[ACTION] Sharing IOCs with MISP...")
try:
    misp_iocs = []
    for indicator in splunk_indicators:
        if indicator.get('type') in ['file', 'user']:
            misp_iocs.append({
                'type': indicator['type'],
                'value': indicator['value'],
                'tags': ['data-collection', 'incident-' + str(incident_id)]
            })

    if misp_iocs:
        misp.share_iocs(misp_iocs, f"Data Collection Incident {incident_id}")
        print(f"   Shared {len(misp_iocs)} IOCs with MISP community")
    else:
        print(f"   No new IOCs to share with MISP")
except Exception as e:
    print(f"   MISP IOC sharing failed: {e}")

# Close IRIS case
print(f"\n[ACTION] Closing IRIS case...")
try:
    iris.close_case(incident_id, 'resolved')
    print(f"   Closed IRIS case {incident_id} as resolved")
except Exception as e:
    print(f"   IRIS case closure failed: {e}")

# Final status summary
print(f"\n Post-incident activities complete:")
print(f"  - IRIS case {incident_id} updated and closed")
print(f"  - Incident report generated")
print(f"  - {len(misp_iocs) if 'misp_iocs' in locals() else 0} IOCs shared with threat intelligence community")
print(f"  - All 5 IR phases completed successfully")

# Calculate incident metrics
incident_duration = (datetime.now() - datetime.fromisoformat(detection_time)).total_seconds() / 3600
print(f"\n📊 Incident Metrics:")
print(f"  - Duration: {incident_duration:.2f} hours")
print(f"  - Systems affected: {len(affected_systems)}")
print(f"  - Containment time: {'< 1 hour' if incident_duration < 1 else f'{incident_duration:.1f} hours'}")
print(f"  - Eradication success: {len([v for v in verification_results if v['status'] == 'clean'])}/{len(affected_systems)} systems clean")
print(f"  - Recovery success: {len([f for f in functional_tests if f['tests_passed'] == f['total_tests']])}/{len(affected_systems)} systems fully functional")


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
